### Feature Engineering

This notebook performs feature engineering to transform cleaned NYC 311 service request data into a format suitable for machine learning models.

The goal of this stage is to create predictive features that help determine whether a complaint will be **resolved within one week**.

Feature engineering steps include:

• Encoding categorical variables  
• Creating time-based indicators  
• Preparing geographic variables  
• Building a modeling-ready feature matrix  
• Splitting the data into training and testing sets  
• Exporting processed datasets for modeling

This step bridges the gap between raw data preparation and predictive modeling.

#### Import libraries
The required Python libraries are imported for feature engineering and model preparation.

Key libraries used:

- **pandas / numpy** for data manipulation
- **scikit-learn preprocessing tools** for encoding categorical variables
- **train_test_split** for creating training and testing datasets

These tools enable transformation of raw features into a modeling-ready format.

In [1]:
# Core analysis libraries
import pandas as pd
import numpy as np

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Improve plot styling
sns.set()

# Show all columns during EDA
pd.set_option('display.max_columns', None)

#### Import data
The cleaned dataset generated in the preprocessing notebook is loaded.

Using the preprocessed dataset ensures that:

• leakage-prone features have already been removed  
• geographic information has been standardized  
• the target variable has already been created  

The focus of this notebook is therefore **feature construction and model preparation**.

In [2]:
# File paths
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent
DATA_DIR = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "models"
SRC_DIR = PROJECT_ROOT / "src"
DEPLOYMENT_DIR = PROJECT_ROOT / "deployment"

In [3]:
# Load data parquet file
data = pd.read_parquet(DATA_DIR / "02_nyc_info_exploratory_analysis.parquet")

In [4]:
# Show first five rows of data
data.head(5)

,unique_key,agency,complaint_type,descriptor,incident_zip,borough,latitude,longitude,location_type,resolution_time_days,resolution_in_wk,complaint_hr,complaint_day,complaint_month
0,68336154,DOT,Street Condition,Pothole,11420,queens,40.678150,-73.831082,unknown,0.000000,0,1,0,3
1,68332725,NYPD,Noise - Residential,Banging/Pounding,11104,queens,40.742897,-73.925399,Residential Building/House,0.010498,0,1,0,3
2,68338535,NYPD,Blocked Driveway,Partial Access,11429,queens,40.715043,-73.737347,Street/Sidewalk,0.016169,0,1,0,3
3,68339824,NYPD,Noise - Residential,Banging/Pounding,10462,bronx,40.838182,-73.859103,Residential Building/House,0.018588,0,1,0,3
4,68335021,NYPD,Noise - Residential,Banging/Pounding,10462,bronx,40.838182,-73.859103,Residential Building/House,0.022338,0,1,0,3


#### Details of data

In [5]:
# Number of rows and columns
data.shape

(263077, 14)

In [6]:
# Data types along with count of missing
types_null_counts = pd.DataFrame()
types_null_counts["Data Type"] = data.dtypes
types_null_counts["Count of Nulls"] = data.isna().sum()
types_null_counts

,Data Type,Count of Nulls
unique_key,object,0
agency,object,0
complaint_type,object,0
descriptor,object,0
incident_zip,object,0
borough,object,0
latitude,float64,0
longitude,float64,0
location_type,object,0
resolution_time_days,float64,0


#### Drop unneeded or leaky columns

In [7]:
# Remove "resolution_time_days" to avoid leakiness
data = data.drop(columns="resolution_time_days").copy()

In [8]:
# Number of rows, columns after drop
data.shape

(263077, 13)

In [9]:
# Review names of all columns
data.columns

Index(['unique_key', 'agency', 'complaint_type', 'descriptor', 'incident_zip',
       'borough', 'latitude', 'longitude', 'location_type', 'resolution_in_wk',
       'complaint_hr', 'complaint_day', 'complaint_month'],
      dtype='object')

#### Target selection
The target variable represents whether a complaint was resolved within seven days.

This is treated as a **binary classification problem**:

• **1** → complaint resolved within one week  
• **0** → complaint resolved after one week  

Separating the target variable from the feature set ensures that model training uses only predictor variables.

In [10]:
# Target selection
y = data["resolution_in_wk"]

#### Feature Selection
Relevant predictor variables are selected for the modeling dataset.

These include features describing:

• complaint characteristics  
• geographic location  
• time of submission  
• responsible agency  

Columns that serve only as identifiers or administrative metadata are excluded from the modeling feature set.

In [11]:
# Continuous features
continuous_features = [
    "latitude",
    "longitude"
]

#### Categorical Variables
Several features in the dataset are categorical, including:

• complaint type  
• descriptor  
• agency  
• borough  
• location type  

These variables are transformed into numerical representations using **one-hot encoding** so that machine learning algorithms can interpret them.

One-hot encoding creates binary indicator variables for each category while avoiding ordinal assumptions.

The feature matrix **X** contains all predictor variables used by the model, while **y** contains the target variable.

Separating predictors from the target variable ensures the model learns patterns only from explanatory features.

In [12]:
# All categorical features
all_categorical_features = [
    "descriptor",
    "incident_zip",
    "agency",
    "complaint_type",
    "borough", 
    "location_type",
    "complaint_hr", 
    "complaint_day", 
    "complaint_month"    
]

In [13]:
# Check number of unique values for all features
for a in all_categorical_features:
    print(a, ":", data[a].nunique())

descriptor : 642
incident_zip : 195
agency : 13
complaint_type : 139
borough : 5
location_type : 84
complaint_hr : 24
complaint_day : 7
complaint_month : 2


In [14]:
# High cardinality features. These should be dropped. 'descriptor' can be nested under 'complaint_type'. 
# Drop 'incident_zip' as it's redundant; we already have longitude and latitude.
high_card_features = [
    "descriptor",
    "incident_zip"
]

In [15]:
# Low cardinality features.
# Drop "complaint_month" for this version of model. 
# High temporal bias is likely since data pulled looks back only 30 days, resulting in only 1-2 months present.
low_card_features = [
    "agency",
    "complaint_type",
    "borough", 
    "location_type",
    "complaint_hr", 
    "complaint_day"
]

In [16]:
# Combine features into one object
model_features = continuous_features + low_card_features
X = data[model_features].copy()

In [17]:
# Confirm X has retained attributes
X.shape

(263077, 8)

#### Train/test split
The dataset is divided into training and testing sets.

The training set is used to train the machine learning models, while the test set is reserved for evaluating model performance on unseen data.

Using a holdout test set ensures that model evaluation reflects real-world predictive performance and avoids overfitting.

In [18]:
# Split data into training and testing slices
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y,
    stratify=y,
    test_size=0.2,
    random_state=42
)

#### Build preprocessor
Categorical encoders are fitted only on the **training data** and then applied to the test data.

This prevents **data leakage**, ensuring that information from the test set does not influence the feature encoding process.

The transformation process therefore follows this sequence:

1. Fit encoder on training data  
2. Transform training data  
3. Apply the same transformation to test data

In [19]:
# Import relevant libraries
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [20]:
# Create preprocessor object to automate tranformations
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), continuous_features),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), low_card_features),
    ],
    verbose_feature_names_out=False
)

In [21]:
# Fit and transform to training data
X_train_processed = preprocessor.fit_transform(X_train)
X_train_processed.shape

(210461, 269)

In [22]:
# Transform to testing data
X_test_processed = preprocessor.transform(X_test)
X_test_processed.shape

(52616, 269)

#### Recreate dataframes
After encoding, the transformed feature matrices are converted back into pandas DataFrames.

This allows:

• easier inspection of feature names  
• compatibility with downstream analysis  
• clearer interpretation of model inputs

In [23]:
feature_names = preprocessor.get_feature_names_out()

X_train_df = pd.DataFrame(X_train_processed, columns=feature_names, index=X_train.index)
X_test_df = pd.DataFrame(X_test_processed, columns=feature_names, index=X_test.index)

#### Export data
The processed training and testing datasets are exported for use in the modeling notebook.

Separating preprocessing, feature engineering, and modeling into different notebooks improves:

• reproducibility  
• modular workflow design  
• clarity when reviewing the machine learning pipeline

In [24]:
X_train_df.to_parquet(DATA_DIR / "03_X_train_processed.parquet")
X_test_df.to_parquet(DATA_DIR / "03_X_test_processed.parquet")

In [25]:
y_train.to_frame().to_parquet(DATA_DIR / "03_y_train.parquet")
y_test.to_frame().to_parquet(DATA_DIR / "03_y_test.parquet")

#### Save to job library

In [26]:
import joblib

joblib.dump(preprocessor, MODELS_DIR / "preprocessor.joblib")

['C:\\Users\\jac67\\Documents\\Data and Analytics\\Python\\nyc-311-project\\models\\preprocessor.joblib']

#### Feature Engineering Summary

The feature engineering process produced a modeling-ready dataset with:

• encoded categorical variables  
• structured geographic features  
• time-based indicators  
• a clean binary classification target

The final output consists of:

• training feature matrix  
• testing feature matrix  
• training target vector  
• testing target vector

These datasets are used in the next notebook to train and evaluate machine learning models predicting whether NYC 311 complaints will be resolved within one week.